### 1: Imports

In [1]:
import pandas as pd
import numpy as np
import pickle
import wandb
from pathlib import Path

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, f1_score

In [3]:
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [4]:
OUTPUT_DIR = Path('../outputs')
DATA_DIR   = Path('../data')

###   2:   MAP@3 + HELPERS FUNCTION

In [5]:
ANSWER_MAP  = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
REVERSE_MAP = {v: k for k, v in ANSWER_MAP.items()}

In [6]:
"""
MAP@3: pos 1→1.0, 
       pos 2→0.5, 
       pos 3→0.33,
       miss→0.0
"""
def map_at_3(y_true, y_proba):
    scores = []
    for i, true in enumerate(y_true):
        top3 = np.argsort(y_proba[i])[-3:][::-1]
        score = (1.0 / (np.where(top3 == true)[0][0] + 1) if true in top3 else 0.0) 
        scores.append(score)
    return float(np.mean(scores))

In [7]:
def make_submission(model, vectorizer, test_df, test_ids):
    X = vectorizer.transform(test_df['combined_text'].values)
    proba = model.predict_proba(X)
    preds = []
    for i in range(len(test_df)):
        top3 = np.argsort(proba[i])[-3:][::-1]
        preds.append(' '.join([REVERSE_MAP[j] for j in top3]))
    return pd.DataFrame({'ID': test_ids, 
                         'Prediction': preds})

In [9]:
def evaluate(model, vectorizer, df, split_name='val'):
    X     = vectorizer.transform(df['combined_text'].values)
    y     = df['answer'].map(ANSWER_MAP).values
    pred  = model.predict(X)
    proba = model.predict_proba(X)
    acc   = accuracy_score(y, pred)
    f1    = f1_score(y, pred, average='weighted')
    m3    = map_at_3(y, proba)
    print(f'  [{split_name}] ACC={acc:.4f}  F1={f1:.4f}  MAP@3={m3:.4f}')
    return {'accuracy': acc, 'f1': f1, 'map_at_3': m3}

print(" Helpers ready")

 Helpers ready
